# Modelos de referencia y experimentos iniciales

Este cuaderno corresponde a la segunda etapa del proyecto y a la primera parte del objetivo específico 2: implementar el flujo de validación, entrenar los modelos de referencia y realizar los experimentos iniciales que ponen a prueba las hipótesis formuladas en el análisis exploratorio. La búsqueda completa de modelos, con las formulaciones ordinales y los demás algoritmos, se realiza en la etapa siguiente.

Las hipótesis provienen de las secciones 8 a 10 del análisis exploratorio:

| Hipótesis | Enunciado | Comparación |
|---|---|---|
| H1 | El modelo de referencia supera al azar | QWK del conjunto de referencia frente a su distribución con etiquetas permutadas |
| H2 | El conjunto principal alcanza un QWK mayor que el de referencia, con una ganancia que se espera pequeña | Principal frente a referencia |
| H3 | Si el conjunto principal mejora, la mejora proviene de la actigrafía | Principal frente a principal sin actigrafía |
| H4 | El conjunto complementario mejora frente al principal | Complementario frente a principal |

Para que las diferencias reflejen la información que aporta cada conjunto y no el algoritmo, todas las comparaciones usan las mismas dos configuraciones fijas, una regresión logística multinomial y un modelo de gradient boosting, sobre los mismos participantes y las mismas particiones. El cuaderno sigue este orden: datos y conjuntos de variables, esquema de validación, métricas y controles, preprocesamiento, modelo de referencia, experimentos iniciales, desempeño por nivel de severidad y conclusión.

Como en el análisis exploratorio, todas las salidas son agregadas. Las predicciones por participante se usan solo para calcular métricas y no se muestran.

In [19]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.dummy import DummyClassifier

from piu_severity.config import FIGURES_DIR, RANDOM_SEED
from piu_severity.data.clean import clean_values
from piu_severity.data.load import list_actigraphy_ids, load_train
from piu_severity.evaluation.metrics import SII_LEVELS, as_levels
from piu_severity.evaluation.validation import (
    N_OUTER_REPEATS,
    N_OUTER_SPLITS,
    aggregate_repeats,
    inner_cv,
    outer_splits,
    run_nested_cv,
    split_level_counts,
    summarize_by_repeat,
)
from piu_severity.features.actigraphy import (
    ACTIGRAPHY_FEATURES,
    load_actigraphy_summary,
)
from piu_severity.features.build import build_analytic_dataset
from piu_severity.features.groups import (
    TARGET_COLUMN,
    column_group,
    experiment_feature_sets,
    is_excluded,
    is_instrument_season,
)

In [20]:
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

FIGURAS = FIGURES_DIR / "02_experimentos_iniciales"

NIVELES_SII = {0: "ninguno", 1: "leve", 2: "moderado", 3: "severo"}
NOMBRES_CONJUNTOS = {
    "reference": "referencia",
    "main": "principal",
    "main_without_actigraphy": "principal sin actigrafía",
    "complementary": "complementario",
}

## 1. Datos y conjuntos de variables

El conjunto de datos analítico se construye con el paquete del proyecto, con los mismos pasos de la sección 10 del análisis exploratorio: depuración de los valores inválidos y recorte a los máximos del protocolo de FitnessGram, fusión de los totales de PAQ-A y PAQ-C e incorporación de las variables de actigrafía de las series utilizables. Como estas reglas son fijas y no se ajustan con los datos, se aplican una sola vez, antes de la validación cruzada.

A partir de esa tabla se definen los cuatro conjuntos de variables de los experimentos: los tres conjuntos de la formulación y el conjunto principal sin las variables de actigrafía, que permite aislar su aporte para la hipótesis H3. Se verifica que los conjuntos estén anidados, que no incluyan el identificador ni las columnas del PCIAT, y que su composición coincida con la del análisis exploratorio.

In [21]:
train = load_train()
etiquetados = train[train[TARGET_COLUMN].notna()].reset_index(drop=True)

analitico = build_analytic_dataset(
    clean_values(etiquetados), load_actigraphy_summary(list_actigraphy_ids())
)
y = as_levels(analitico[TARGET_COLUMN])
conjuntos = experiment_feature_sets(analitico.columns)
columnas_actigrafia = list(ACTIGRAPHY_FEATURES.values())

assert len(analitico) == len(etiquetados)
assert analitico["id"].is_unique
assert not any(
    is_excluded(c) or is_instrument_season(c)
    for columnas in conjuntos.values()
    for c in columnas
)
assert (
    set(conjuntos["reference"])
    <= set(conjuntos["main_without_actigraphy"])
    <= set(conjuntos["main"])
    <= set(conjuntos["complementary"])
)
assert set(conjuntos["main"]) - set(conjuntos["main_without_actigraphy"]) == set(
    columnas_actigrafia
)

distribucion_sii = (
    pd.Series(y)
    .map(NIVELES_SII)
    .value_counts()
    .reindex(NIVELES_SII.values())
    .rename_axis("SII")
    .to_frame("participantes")
)
distribucion_sii["%"] = (
    100 * distribucion_sii["participantes"] / distribucion_sii["participantes"].sum()
).round(1)
distribucion_sii

,participantes,%
SII,,
ninguno,1594,58.3
leve,730,26.7
moderado,378,13.8
severo,34,1.2


In [22]:
GRUPOS_ES = {
    "demographic": "demográficas",
    "physical": "físicas",
    "complementary": "complementarias",
}

composicion_conjuntos = pd.DataFrame(
    {
        NOMBRES_CONJUNTOS[nombre]: pd.Series(
            [GRUPOS_ES[column_group(c)] for c in columnas]
        ).value_counts()
        for nombre, columnas in conjuntos.items()
    }
).reindex(GRUPOS_ES.values()).fillna(0).astype(int).T
composicion_conjuntos["total"] = composicion_conjuntos.sum(axis="columns")
composicion_conjuntos["incluye actigrafía"] = [
    "sí" if set(columnas_actigrafia) <= set(columnas) else "no"
    for columnas in conjuntos.values()
]
composicion_conjuntos.index.name = "conjunto"

print(f"Participantes etiquetados: {len(analitico)}")
print(
    "Participantes con variables de actigrafía: "
    f"{int(analitico[columnas_actigrafia].notna().any(axis='columns').sum())}"
)
composicion_conjuntos

Participantes etiquetados: 2736
Participantes con variables de actigrafía: 823


,demográficas,físicas,complementarias,total,incluye actigrafía
conjunto,,,,,
referencia,3,0,0,3,no
principal,3,33,0,36,sí
principal sin actigrafía,3,30,0,33,no
complementario,3,33,3,39,sí


### Conclusión

El conjunto de datos analítico reproduce el del análisis exploratorio: contiene a los 2.736 participantes etiquetados, de los cuales 823 tienen variables de actigrafía, y conserva la distribución del SII (58,3 % ninguno, 26,7 % leve, 13,8 % moderado y 1,2 % severo, con 34 casos severos).

| Conjunto | Demográficas | Físicas | Complementarias | Total | Incluye actigrafía |
|---|---|---|---|---|---|
| Referencia | 3 | 0 | 0 | 3 | No |
| Principal sin actigrafía | 3 | 30 | 0 | 33 | No |
| Principal | 3 | 33 | 0 | 36 | Sí |
| Complementario | 3 | 33 | 3 | 39 | Sí |

Cada fila indica cuántas variables de cada grupo contiene el conjunto. Los cuatro conjuntos están anidados en el orden de la tabla y ninguno incluye el identificador, las columnas del PCIAT ni las temporadas de instrumento. Con esta estructura, cada hipótesis compara dos conjuntos que difieren en un solo bloque de variables: H2, todas las variables físicas; H3, solo la actigrafía; y H4, las variables complementarias.

## 2. Esquema de validación

Cada modelo requiere dos decisiones que deben tomarse con datos distintos: elegir sus hiperparámetros, como la intensidad de la regularización, y medir su desempeño en participantes que no vio durante el entrenamiento. Si ambas se hicieran con los mismos datos, el desempeño estimado sería optimista, porque los hiperparámetros elegidos serían los que mejor se ajustan precisamente a esos participantes. Para separarlas se usa validación cruzada anidada, que tiene dos niveles.

**Nivel externo: medición del desempeño.** Los 2.736 participantes se dividen en cinco partes de tamaño similar, estratificadas por SII para que cada parte conserve la proporción de cada nivel de severidad. Se hacen cinco vueltas: en cada una, una de las partes se aparta como partición de evaluación y las otras cuatro forman la partición de entrenamiento. Al terminar, cada participante se ha evaluado exactamente una vez.

**Nivel interno: elección de hiperparámetros.** Dentro de cada partición de entrenamiento externa, y solo con esos participantes, se repite el mismo procedimiento con tres partes. Cada combinación candidata de hiperparámetros se entrena con dos partes y se evalúa con la tercera, rotando tres veces, y se elige la de mejor QWK promedio. Con esa combinación, el modelo se reentrena con toda la partición de entrenamiento externa y se evalúa en la partición de evaluación externa, que no intervino en ninguna decisión de ajuste.

| Nivel | Qué participantes se dividen | Partes | Participantes para entrenar | Participantes para evaluar | Uso |
|---|---|---|---|---|---|
| Externo | Los 2.736 participantes | 5 | Alrededor de 2.189 (cuatro de las cinco partes) | Alrededor de 547 (una parte) | Medir el desempeño |
| Interno | Los participantes de entrenamiento de una partición externa | 3 | Alrededor de 1.459 (dos de las tres partes) | Alrededor de 730 (una parte) | Elegir los hiperparámetros |

El nivel externo se repite cinco veces, con una división distinta de los participantes en cada repetición; la repetición r usa la semilla base más r. Esto da 25 particiones externas y reduce la dependencia de los resultados respecto de una sola forma de dividir los datos. Como todas las configuraciones y todos los conjuntos de variables se evalúan sobre las mismas 25 particiones, las comparaciones entre ellos se pueden hacer de forma pareada, sobre los mismos participantes.

Esta sección construye las particiones y verifica tres propiedades: que cada participante se evalúe exactamente una vez en cada repetición, que la estratificación mantenga la proporción de cada nivel de SII y cuántos casos de cada nivel contiene cada partición de evaluación, externa e interna. Este último conteo condiciona la estabilidad de las métricas del nivel severo.

In [23]:
particiones = outer_splits(y)

for repeticion in range(N_OUTER_REPEATS):
    evaluados = np.concatenate([p.test for p in particiones if p.repeat == repeticion])
    assert np.array_equal(np.sort(evaluados), np.arange(len(y)))
assert all(np.intersect1d(p.train, p.test).size == 0 for p in particiones)

tamanos_evaluacion = pd.Series([len(p.test) for p in particiones])
print(
    f"Particiones externas: {len(particiones)} "
    f"({N_OUTER_SPLITS} particiones × {N_OUTER_REPEATS} repeticiones)"
)
print(
    "Participantes por partición externa de evaluación: "
    f"entre {tamanos_evaluacion.min()} y {tamanos_evaluacion.max()}"
)

Particiones externas: 25 (5 particiones × 5 repeticiones)
Participantes por partición externa de evaluación: entre 547 y 548


In [24]:
casos_externos = split_level_counts(particiones, y)

filas_internas = []
for particion in particiones:
    y_entrenamiento = y[particion.train]
    for _, evaluacion_interna in inner_cv().split(
        np.zeros(len(y_entrenamiento)), y_entrenamiento
    ):
        conteo = pd.Series(y_entrenamiento[evaluacion_interna]).value_counts()
        filas_internas.append(conteo.reindex(SII_LEVELS, fill_value=0))
casos_internos = pd.DataFrame(filas_internas)

casos_por_particion = pd.concat(
    {
        "partición externa": casos_externos.agg(["min", "mean", "max"]).T,
        "partición interna": casos_internos.agg(["min", "mean", "max"]).T,
    },
    axis="columns",
).rename(index=NIVELES_SII, columns={"min": "mínimo", "mean": "media", "max": "máximo"})
casos_por_particion.index.name = "SII"

print(f"Particiones internas de evaluación: {len(casos_internos)}")
casos_por_particion.round(1)

Particiones internas de evaluación: 75


partición externa               partición interna              
                    mínimo  media máximo            mínimo  media máximo
SII                                                                     
ninguno              318.0  318.8  319.0             425.0  425.1  426.0
leve                 146.0  146.0  146.0             194.0  194.7  195.0
moderado              75.0   75.6   76.0             100.0  100.8  101.0
severo                 6.0    6.8    7.0               9.0    9.1   10.0

### Conclusión

#### Verificación de las particiones

Se generaron 25 particiones externas: cinco particiones en cada una de las cinco repeticiones. Cumplen las dos verificaciones: en cada repetición, cada participante se evalúa exactamente una vez, y ninguna partición comparte participantes entre entrenamiento y evaluación. Cada partición externa de evaluación contiene 547 o 548 participantes.

#### Casos de cada nivel por partición

Hay dos tipos de partición de evaluación. Las **externas** (25) miden el desempeño final de cada modelo. Las **internas** (75, tres dentro de cada partición externa) se usan solo para elegir los hiperparámetros, y se forman con los participantes de entrenamiento de la partición externa.

| SII | Casos en cada partición externa | Casos en cada partición interna |
|---|---|---|
| Ninguno | 318 o 319 | Entre 425 y 426 |
| Leve | 146 | Entre 194 y 195 |
| Moderado | 75 o 76 | Entre 100 y 101 |
| Severo | 6 o 7 | Entre 9 y 10 |

La tabla se lee por filas: por ejemplo, cada partición externa contiene 6 o 7 participantes con SII severo. En todos los niveles, el número de casos varía como máximo en uno entre particiones, de modo que la estratificación es prácticamente exacta y las diferencias de desempeño entre particiones no se deben a cambios en la proporción de los niveles.

#### Consecuencias para la evaluación

El nivel severo es el que limita la precisión de las estimaciones. Con 6 o 7 casos por partición externa, la sensibilidad de ese nivel calculada en una sola partición solo puede variar en saltos de alrededor de 14 puntos porcentuales: detectar un caso más o un caso menos cambia el resultado en esa magnitud. Por eso las métricas se calculan de dos formas:

- **QWK, por partición.** Así se obtienen 25 valores que permiten comparar dos configuraciones de forma pareada, sobre los mismos participantes.
- **Sensibilidad por nivel, tasa de subestimación y matriz de confusión, sobre las predicciones acumuladas de cada repetición.** Como en cada repetición todos los participantes se evalúan una vez, estas métricas usan los 34 casos severos y no solo los 6 o 7 de una partición.

En la validación interna, cada partición de evaluación conserva 9 o 10 casos severos. La selección de hiperparámetros se apoya, por tanto, en más casos de los que tendría con cinco particiones internas (alrededor de 5), aunque sigue dependiendo de pocos participantes de ese nivel.

## 3. Métricas y controles de cordura

El desempeño se mide con cuatro métricas, definidas en la formulación del proyecto:

- **QWK (coeficiente kappa ponderado cuadrático).** Mide la concordancia entre el nivel real y el estimado, y penaliza cada error en proporción al cuadrado de la distancia entre ambos niveles. Vale 1 con concordancia perfecta y 0 cuando la concordancia equivale a la esperada por azar, dadas las distribuciones de los niveles reales y estimados. Es la métrica principal.
- **Sensibilidad por nivel.** Proporción de participantes de cada nivel que el modelo asigna a ese mismo nivel.
- **Tasa de subestimación.** Proporción de participantes con PIU moderado o severo a los que se asigna un nivel inferior al real. Se reporta también solo para el nivel severo.
- **Matriz de confusión.** Distribución de los niveles estimados para cada nivel real.

Como se definió en la sección anterior, el QWK se calcula en cada partición externa y se resume con la media de las particiones de cada repetición; las demás métricas se calculan sobre las predicciones acumuladas de cada repetición. Todas se reportan como media y desviación estándar entre las cinco repeticiones.

Antes de entrenar modelos, el flujo de evaluación se verifica con dos clasificadores que no usan los predictores:

- **Clase mayoritaria.** Asigna a todos los participantes el nivel más frecuente en la partición de entrenamiento. Por construcción, su QWK es 0.
- **Aleatorio estratificado.** Asigna niveles al azar con las proporciones de la partición de entrenamiento. Su QWK esperado también es 0, aunque varía entre particiones.

Estos dos clasificadores cumplen dos funciones. Verifican que el cálculo de las métricas sobre las particiones produzca los valores esperados para un modelo sin información, y constituyen el primer nivel de la jerarquía de modelos de referencia: cualquier modelo útil debe superarlos.

In [25]:
CONTROLES = {
    "clase mayoritaria": DummyClassifier(strategy="most_frequent"),
    "aleatorio estratificado": DummyClassifier(
        strategy="stratified", random_state=RANDOM_SEED
    ),
}
X_referencia = analitico[conjuntos["reference"]]

resultados_controles = {
    nombre: run_nested_cv(modelo, X_referencia, y, particiones)
    for nombre, modelo in CONTROLES.items()
}

In [26]:
METRICAS_ES = {
    "qwk": "QWK",
    "recall_0": "sensibilidad ninguno",
    "recall_1": "sensibilidad leve",
    "recall_2": "sensibilidad moderado",
    "recall_3": "sensibilidad severo",
    "underestimation": "subestimación moderado o severo",
    "underestimation_severe": "subestimación severo",
}


def tabla_desempeno(resultados: dict) -> pd.DataFrame:
    """Resume cada modelo con la media y la desviación entre repeticiones."""
    filas = {}
    for nombre, (pliegues, predicciones) in resultados.items():
        agregado = aggregate_repeats(summarize_by_repeat(pliegues, predicciones))
        filas[nombre] = {
            (METRICAS_ES[metrica], estadistico): agregado.at[metrica, columna]
            for metrica in METRICAS_ES
            for estadistico, columna in [("media", "mean"), ("desv.", "std")]
        }
    tabla = pd.DataFrame(filas).T
    tabla.columns = pd.MultiIndex.from_tuples(tabla.columns)
    return tabla


tabla_controles = tabla_desempeno(resultados_controles)
qwk_por_particion = pd.DataFrame(
    {nombre: pliegues["qwk"] for nombre, (pliegues, _) in resultados_controles.items()}
).agg(["min", "max"]).T.rename(columns={"min": "QWK mínimo", "max": "QWK máximo"})

display(tabla_controles.round(3))
qwk_por_particion.round(3)

QWK        sensibilidad ninguno         \
                        media  desv.                media  desv.   
clase mayoritaria        0.00  0.000                1.000  0.000   
aleatorio estratificado -0.01  0.022                0.586  0.011   

                        sensibilidad leve        sensibilidad moderado         \
                                    media  desv.                 media  desv.   
clase mayoritaria                   0.000  0.000                 0.000  0.000   
aleatorio estratificado             0.273  0.021                 0.123  0.025   

                        sensibilidad severo         \
                                      media  desv.   
clase mayoritaria                     0.000  0.000   
aleatorio estratificado               0.006  0.013   

                        subestimación moderado o severo         \
                                                  media  desv.   
clase mayoritaria                                 1.000  0.000   
aleatorio estratificado                           0.874  0.022   

                        subestimación severo         
                                       media  desv.  
clase mayoritaria                      1.000  0.000  
aleatorio estratificado                0.994  0.013

,QWK mínimo,QWK máximo
clase mayoritaria,0.000,0.000
aleatorio estratificado,-0.078,0.082


### Conclusión

Los dos controles producen los valores esperados para clasificadores sin información, lo que confirma que el cálculo de las métricas sobre las particiones funciona correctamente.

| Métrica | Clase mayoritaria | Aleatorio estratificado |
|---|---|---|
| QWK | 0,000 ± 0,000 | −0,010 ± 0,022 |
| QWK por partición (mínimo y máximo) | 0,000 y 0,000 | −0,078 y 0,082 |
| Sensibilidad ninguno | 1,000 | 0,586 |
| Sensibilidad leve | 0,000 | 0,273 |
| Sensibilidad moderado | 0,000 | 0,123 |
| Sensibilidad severo | 0,000 | 0,006 |
| Subestimación moderado o severo | 1,000 | 0,874 |
| Subestimación severo | 1,000 | 0,994 |

Los valores con ± son la media y la desviación estándar entre las cinco repeticiones. Las sensibilidades y las tasas de subestimación se muestran solo con la media.

- **Clase mayoritaria.** Asigna el nivel ninguno a todos los participantes, por lo que su sensibilidad es 1 en ese nivel y 0 en los demás, subestima a todos los participantes con PIU moderado o severo y su QWK es exactamente 0 en las 25 particiones.
- **Aleatorio estratificado.** Su sensibilidad en cada nivel coincide con la proporción de ese nivel en los datos (58,3 %, 26,7 %, 13,8 % y 1,2 %), como corresponde a asignaciones al azar con esas proporciones. Su QWK medio es prácticamente 0.

El control aleatorio establece además el margen de variación que produce el azar. Sin ninguna información, el QWK de una sola partición varió entre −0,078 y 0,082. Por tanto, una diferencia de esa magnitud entre dos modelos en una partición aislada no indica por sí sola una mejora; por esta razón, las comparaciones entre configuraciones se basan en las 25 diferencias pareadas y no en particiones individuales.

Por último, ambos controles muestran que la tasa de subestimación es alta cuando un modelo no discrimina: un modelo sin información subestima a casi 9 de cada 10 participantes con PIU moderado o severo, porque la mayoría de sus asignaciones corresponden a los niveles más frecuentes. Este es el punto de partida frente al cual se juzga la subestimación de los modelos entrenados, y justifica la ponderación de las clases durante el entrenamiento.